In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import numpy as np
from datetime import datetime, timedelta

In [ ]:
from aquacrop.utils import prepare_weather, get_filepath

In [ ]:
import rasterio as rio
import geopandas as gpd

In [ ]:
# repository root (notebooks live in <repo>/notebooks)
base_loc = os.path.abspath(os.path.join(os.getcwd(), '..'))
weather_loc=os.path.join(base_loc,'data','weather_aquacrop')
output_loc=os.path.join(base_loc,'output')

In [ ]:
def get_weather(dist):
    filepath=get_filepath(os.path.join(weather_loc,dist+'.txt'))
    weather_data = prepare_weather(filepath)
    return weather_data

In [ ]:
# Rajasthan district boundaries (subset of the all-India district shapefile)
india_shape = os.path.join(base_loc, 'data', 'location data', 'Rajasthan.shp')
data=gpd.read_file(india_shape)
raj=data[data['STATE_NAME']=='Rajasthan']
raj=raj.drop(columns=['Crop','ID'], errors='ignore')

In [ ]:
districts=list(raj['DISTRICT'].unique())

In [ ]:
temp=pd.DataFrame(index=['MinT','MaxT','ET','Rainfall'],columns=districts)
rf=pd.DataFrame()
for d in districts:
    df=get_weather(d)
    df['Year']=df['Date'].apply(lambda x:x.year)
    df=df.rename(columns={'Date':d})
    df=df.set_index(d)
    rain=df.copy()
    rain=rain.drop(columns=['MinTemp','MaxTemp','ReferenceET'])
    rain['Precipitation']=np.where(rain['Precipitation']==-999,np.nan,rain['Precipitation'])
    test=rain.groupby('Year').sum().mean()
    df=df.drop(columns=['Year','Precipitation'])
    x=list(round(df.mean(),2))
    temp.loc['MinT'][d]=x[0]
    temp.loc['MaxT'][d]=x[1]
    temp.loc['ET'][d]=x[2]
    temp.loc['Rainfall'][d]=round(test[0],2)

In [ ]:
weather_data=temp.transpose()
weather_data['DISTRICT']=weather_data.index
weather_data=weather_data.set_index('DISTRICT')

In [ ]:
cv=pd.DataFrame()
meanYield=pd.DataFrame()
for d in np.arange(1,8):
    req=pd.DataFrame()
    d=int((d-1)*15)
    for dist in districts:
        
        init_date=datetime.strptime('01-06','%d-%m')
        date=init_date+timedelta(days=d)
        planting_date=(datetime.strftime(date,'%B %d'))
        plt_dt=datetime.strftime(date,'%m-%d')
        fn=os.path.join(base_loc,'output','Final_stats_'+plt_dt+'-'+dist+'.xlsx')
        df=pd.read_excel(fn)
        req[dist]=df['Yield (tonne/ha)']
    cv[planting_date]=req.std()/req.mean()
    meanYield[planting_date]=req.mean()

In [ ]:
cv=round(cv,2)
cv['DISTRICT']=cv.index
cv=cv.set_index('DISTRICT')

In [ ]:
meanYield=round(meanYield,2)
meanYield['DISTRICT']=meanYield.index
meanYield=meanYield.set_index('DISTRICT')

In [ ]:
raj_cv=raj.merge(cv,on='DISTRICT')
raj_mean=raj.merge(meanYield,on='DISTRICT')
raj_wet=raj.merge(weather_data,on='DISTRICT')

In [ ]:
raj_cv.to_file(os.path.join(base_loc,'data','location data','Rajasthan_PM_CV.shp'))
raj_mean.to_file(os.path.join(base_loc,'data','location data','Rajasthan_PM_mean.shp'))
raj_wet.to_file(os.path.join(base_loc,'data','location data','Rajasthan_PM_weather.shp'))